# Model Selection

Dieses Notebook vergleicht die festgelegten Modellfamilien auf dem zeitlichen Split: Training von November 2025 bis Juni 2026, Validierung im Juli 2026.


## 1. Imports und Einstellungen

Es werden nur die Bibliotheken importiert, die im Notebook direkt verwendet werden.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from pygam import LinearGAM, f, l, s
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor


def evaluate_predictions(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


pd.set_option("display.max_columns", 50)

PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / "datasets").exists():
    if PROJECT_DIR == PROJECT_DIR.parent:
        raise FileNotFoundError("Projektordner mit datasets/ wurde nicht gefunden.")
    PROJECT_DIR = PROJECT_DIR.parent

FEATURE_DIR = PROJECT_DIR / "datasets" / "modeling" / "features"
TRAIN_PATH = FEATURE_DIR / "train_features.parquet"
VALIDATION_PATH = FEATURE_DIR / "validation_features.parquet"


## 2. Daten laden

Geladen werden nur der Trainingsdatensatz und der Juli-Validierungsdatensatz. Der August-Testdatensatz wird in diesem Notebook nicht gelesen.


In [2]:
train = pd.read_parquet(TRAIN_PATH)
validation = pd.read_parquet(VALIDATION_PATH)

dataset_overview = pd.DataFrame(
    [
        {
            "Datensatz": "Training",
            "Zeitraum": "November 2025 bis Juni 2026",
            "Beobachtungen": len(train),
        },
        {
            "Datensatz": "Validation",
            "Zeitraum": "Juli 2026",
            "Beobachtungen": len(validation),
        },
    ]
)

dataset_overview


,Datensatz,Zeitraum,Beobachtungen
0,Training,November 2025 bis Juni 2026,689684
1,Validation,Juli 2026,73867


## 3. Features und Target definieren

Das Target ist `delay_target`. Das Feature-Set ist auf Kalender- und Wettermerkmale beschränkt.


In [3]:
target_column = "delay_target"

categorical_features = [
    "event_hour",
    "event_weekday",
    "rr_precipitation_form",
]

numeric_features = [
    "is_public_holiday",
    "ff_wind_speed_ms",
    "wind_direction_sin",
    "wind_direction_cos",
    "fx_wind_gust_ms",
    "tu_temperature_c",
    "tu_relative_humidity_pct",
    "rr_precipitation_mm",
    "rr_precipitation_indicator",
]

model_features = categorical_features + numeric_features

y_train = train[target_column]
y_validation = validation[target_column]

X_train = train[model_features].copy()
X_validation = validation[model_features].copy()

feature_overview = pd.DataFrame(
    {
        "Feature": model_features,
        "Datentyp Training": [X_train[column].dtype for column in model_features],
        "Missing Training": [X_train[column].isna().sum() for column in model_features],
        "Missing Validation": [X_validation[column].isna().sum() for column in model_features],
    }
)

feature_overview


,Feature,Datentyp Training,Missing Training,Missing Validation
0,event_hour,int32,0,0
1,event_weekday,int32,0,0
2,rr_precipitation_form,float64,209,0
3,is_public_holiday,bool,0,0
4,ff_wind_speed_ms,float64,4769,0
5,wind_direction_sin,float64,4205,0
6,wind_direction_cos,float64,4205,0
7,fx_wind_gust_ms,float64,4769,0
8,tu_temperature_c,float64,1506,0
9,tu_relative_humidity_pct,float64,1597,0


## 4. Preprocessing

Für sklearn und XGBoost werden kategoriale Merkmale als Strings behandelt und per One-Hot-Encoding kodiert. Numerische Missing Values werden über eine Median-Imputation gelernt, die nur auf den Trainingsdaten gefittet wird. Für Random Forest wird zusätzlich eine dichte Kopie der transformierten Matrix verwendet, weil sklearn-Bäume damit deutlich schneller laufen.


In [4]:
X_train_sklearn = X_train[model_features].copy()
X_validation_sklearn = X_validation[model_features].copy()

for column in categorical_features:
    X_train_sklearn[column] = X_train_sklearn[column].fillna("missing").astype(str)
    X_validation_sklearn[column] = X_validation_sklearn[column].fillna("missing").astype(str)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

# Median-Imputation und One-Hot-Encoding werden nur auf Training gelernt.
X_train_processed = preprocessor.fit_transform(X_train_sklearn)
X_validation_processed = preprocessor.transform(X_validation_sklearn)

# Random Forest laeuft mit dichter Matrix deutlich schneller; die Transformation bleibt identisch.
X_train_processed_dense = X_train_processed.toarray()
X_validation_processed_dense = X_validation_processed.toarray()

model_results = []

print(f"Kategoriale Features: {len(categorical_features)}")
print(f"Numerische Features: {len(numeric_features)}")
print(f"Modellfeatures: {len(model_features)}")
print(f"Transformierte Features: {X_train_processed.shape[1]}")


Kategoriale Features: 3
Numerische Features: 9
Modellfeatures: 12
Transformierte Features: 46


## 5. Baselines

Median und Mittelwert des Trainings-Targets dienen als einfache Referenzmodelle.


In [5]:
median_prediction_value = y_train.median()
median_predictions = np.full(len(y_validation), median_prediction_value)
median_metrics = evaluate_predictions(y_validation, median_predictions)
model_results.append({"Model": "Median Baseline", **median_metrics})

mean_prediction_value = y_train.mean()
mean_predictions = np.full(len(y_validation), mean_prediction_value)
mean_metrics = evaluate_predictions(y_validation, mean_predictions)
model_results.append({"Model": "Mean Baseline", **mean_metrics})

baseline_results = pd.DataFrame(
    [
        {"Baseline": "Median", "Konstante Vorhersage": median_prediction_value, **median_metrics},
        {"Baseline": "Mean", "Konstante Vorhersage": mean_prediction_value, **mean_metrics},
    ]
)

baseline_results.round(3)


,Baseline,Konstante Vorhersage,MAE,RMSE,R2
0,Median,2.000,5.628,11.163,-0.174
1,Mean,5.431,6.197,10.340,-0.007


## 6. Lineare Regression

Die lineare Regression nutzt das gemeinsame sklearn-Preprocessing und dient als gut interpretierbares Vergleichsmodell.


In [6]:
linear_regression_model = LinearRegression()

linear_regression_model.fit(X_train_processed, y_train)
linear_regression_predictions = linear_regression_model.predict(X_validation_processed)
linear_regression_metrics = evaluate_predictions(y_validation, linear_regression_predictions)
model_results.append({"Model": "Lineare Regression", **linear_regression_metrics})

pd.DataFrame([linear_regression_metrics]).round(3)


,MAE,RMSE,R2
0,6.311,10.208,0.018


## 7. Generalized Additive Model (GAM)

Für das GAM werden kategoriale Merkmale als Faktoren kodiert. Die Wettervariablen werden mit Splines modelliert, zwei binäre bzw. indikatorartige Merkmale linear.


In [7]:
X_train_gam = X_train[model_features].copy()
X_validation_gam = X_validation[model_features].copy()

for column in categorical_features:
    train_values = X_train_gam[column].fillna("missing").astype(str)
    validation_values = X_validation_gam[column].fillna("missing").astype(str)

    categories = sorted(train_values.unique().tolist())
    if "missing" not in categories:
        categories.append("missing")

    category_mapping = {category: code for code, category in enumerate(categories)}
    X_train_gam[column] = train_values.map(category_mapping).astype(int)
    X_validation_gam[column] = (
        validation_values.where(validation_values.isin(set(category_mapping)), "missing")
        .map(category_mapping)
        .astype(int)
    )

gam_numeric_imputer = SimpleImputer(strategy="median")
X_train_gam[numeric_features] = gam_numeric_imputer.fit_transform(X_train_gam[numeric_features])
X_validation_gam[numeric_features] = gam_numeric_imputer.transform(X_validation_gam[numeric_features])

# model_features: 0 event_hour, 1 event_weekday, 2 rr_precipitation_form,
# 3 is_public_holiday, 4-10 Wetter-Splines, 11 rr_precipitation_indicator.
gam_model = LinearGAM(
    f(0) + f(1) + f(2)
    + l(3)
    + s(4, n_splines=10, lam=1.0)
    + s(5, n_splines=10, lam=1.0)
    + s(6, n_splines=10, lam=1.0)
    + s(7, n_splines=10, lam=1.0)
    + s(8, n_splines=10, lam=1.0)
    + s(9, n_splines=10, lam=1.0)
    + s(10, n_splines=10, lam=1.0)
    + l(11)
)

gam_model.fit(X_train_gam, y_train)
gam_predictions = gam_model.predict(X_validation_gam)
gam_metrics = evaluate_predictions(y_validation, gam_predictions)
model_results.append({"Model": "GAM", **gam_metrics})

gam_parameter_sets = [
    {"n_splines": 8, "lam": 0.6},
    {"n_splines": 8, "lam": 5.0},
    {"n_splines": 12, "lam": 0.6},
    {"n_splines": 12, "lam": 5.0},
]

gam_tuning_rows = []

for params in gam_parameter_sets:
    model = LinearGAM(
        f(0) + f(1) + f(2)
        + l(3)
        + s(4, n_splines=params["n_splines"], lam=params["lam"])
        + s(5, n_splines=params["n_splines"], lam=params["lam"])
        + s(6, n_splines=params["n_splines"], lam=params["lam"])
        + s(7, n_splines=params["n_splines"], lam=params["lam"])
        + s(8, n_splines=params["n_splines"], lam=params["lam"])
        + s(9, n_splines=params["n_splines"], lam=params["lam"])
        + s(10, n_splines=params["n_splines"], lam=params["lam"])
        + l(11)
    )
    model.fit(X_train_gam, y_train)
    train_predictions = model.predict(X_train_gam)
    validation_predictions = model.predict(X_validation_gam)
    validation_metrics = evaluate_predictions(y_validation, validation_predictions)
    gam_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation MAE": validation_metrics["MAE"],
            "Validation RMSE": validation_metrics["RMSE"],
            "Validation R2": validation_metrics["R2"],
        }
    )

gam_tuning_results = (
    pd.DataFrame(gam_tuning_rows)
    .sort_values("Validation R2", ascending=False)
    .reset_index(drop=True)
)

best_gam_params = gam_tuning_results.loc[0, ["n_splines", "lam"]].to_dict()
best_gam_params["n_splines"] = int(best_gam_params["n_splines"])

gam_tuned_model = LinearGAM(
    f(0) + f(1) + f(2)
    + l(3)
    + s(4, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + s(5, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + s(6, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + s(7, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + s(8, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + s(9, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + s(10, n_splines=best_gam_params["n_splines"], lam=best_gam_params["lam"])
    + l(11)
)
gam_tuned_model.fit(X_train_gam, y_train)
gam_tuned_predictions = gam_tuned_model.predict(X_validation_gam)
gam_tuned_metrics = evaluate_predictions(y_validation, gam_tuned_predictions)
model_results.append({"Model": "GAM Tuned", **gam_tuned_metrics})

gam_tuning_results.round(3)


,n_splines,lam,Train R2,Validation MAE,Validation RMSE,Validation R2
0,12,5.0,0.030,6.373,10.246,0.011
1,12,0.6,0.030,6.372,10.246,0.011
2,8,0.6,0.029,6.364,10.252,0.010
3,8,5.0,0.029,6.364,10.253,0.010


## 8. Random Forest

Zuerst wird ein Basismodell trainiert. Danach werden wenige feste Parameterkombinationen auf dem Juli-Validierungsdatensatz verglichen.


In [8]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)

random_forest_model.fit(X_train_processed_dense, y_train)
random_forest_predictions = random_forest_model.predict(X_validation_processed_dense)
random_forest_metrics = evaluate_predictions(y_validation, random_forest_predictions)
model_results.append({"Model": "Random Forest", **random_forest_metrics})

random_forest_parameter_sets = [
    {"n_estimators": 100, "max_depth": 10, "min_samples_leaf": 5},
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 10},
    {"n_estimators": 200, "max_depth": 15, "min_samples_leaf": 5},
    {"n_estimators": 300, "max_depth": 15, "min_samples_leaf": 10},
]

random_forest_tuning_rows = []

for params in random_forest_parameter_sets:
    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_processed_dense, y_train)
    train_predictions = model.predict(X_train_processed_dense)
    validation_predictions = model.predict(X_validation_processed_dense)
    validation_metrics = evaluate_predictions(y_validation, validation_predictions)
    random_forest_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation MAE": validation_metrics["MAE"],
            "Validation RMSE": validation_metrics["RMSE"],
            "Validation R2": validation_metrics["R2"],
        }
    )

random_forest_tuning_results = (
    pd.DataFrame(random_forest_tuning_rows)
    .sort_values("Validation R2", ascending=False)
    .reset_index(drop=True)
)

best_random_forest_params = {
    key: int(value)
    for key, value in random_forest_tuning_results.loc[
        0, ["n_estimators", "max_depth", "min_samples_leaf"]
    ].to_dict().items()
}

random_forest_tuned_model = RandomForestRegressor(
    **best_random_forest_params,
    random_state=42,
    n_jobs=-1,
)

random_forest_tuned_model.fit(X_train_processed_dense, y_train)
random_forest_tuned_predictions = random_forest_tuned_model.predict(X_validation_processed_dense)
random_forest_tuned_metrics = evaluate_predictions(y_validation, random_forest_tuned_predictions)
model_results.append({"Model": "Random Forest Tuned", **random_forest_tuned_metrics})

random_forest_tuning_results.round(3)


,n_estimators,max_depth,min_samples_leaf,Train R2,Validation MAE,Validation RMSE,Validation R2
0,200,10,10,0.049,6.531,10.249,0.011
1,100,10,5,0.049,6.543,10.265,0.007
2,300,15,10,0.083,6.685,10.309,-0.001
3,200,15,5,0.085,6.769,10.358,-0.011


## 9. XGBoost

Basislauf und begrenztes Tuning stehen bewusst in einer Codezelle: Parameter definieren, Modell fitten, Juli vorhersagen, Metriken vergleichen.


In [9]:
xgboost_base_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgboost_base_model.fit(X_train_processed, y_train)
xgboost_predictions = xgboost_base_model.predict(X_validation_processed)
xgboost_metrics = evaluate_predictions(y_validation, xgboost_predictions)
model_results.append({"Model": "XGBoost", **xgboost_metrics})

xgboost_parameter_sets = [
    {"n_estimators": 150, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.03},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.04},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 250, "max_depth": 3, "learning_rate": 0.04},
    {"n_estimators": 250, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.03},
    {"n_estimators": 300, "max_depth": 5, "learning_rate": 0.03},
    {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03},
    {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.03},
]

xgboost_tuning_rows = []

for params in xgboost_parameter_sets:
    model = XGBRegressor(
        **params,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_processed, y_train)
    train_predictions = model.predict(X_train_processed)
    validation_predictions = model.predict(X_validation_processed)
    validation_metrics = evaluate_predictions(y_validation, validation_predictions)
    xgboost_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation MAE": validation_metrics["MAE"],
            "Validation RMSE": validation_metrics["RMSE"],
            "Validation R2": validation_metrics["R2"],
        }
    )

xgboost_tuning_results = (
    pd.DataFrame(xgboost_tuning_rows)
    .sort_values("Validation R2", ascending=False)
    .reset_index(drop=True)
)

best_xgboost_params = xgboost_tuning_results.loc[
    0, ["n_estimators", "max_depth", "learning_rate"]
].to_dict()
best_xgboost_params["n_estimators"] = int(best_xgboost_params["n_estimators"])
best_xgboost_params["max_depth"] = int(best_xgboost_params["max_depth"])

xgboost_tuned_model = XGBRegressor(
    **best_xgboost_params,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

xgboost_tuned_model.fit(X_train_processed, y_train)
xgboost_tuned_train_predictions = xgboost_tuned_model.predict(X_train_processed)
xgboost_tuned_predictions = xgboost_tuned_model.predict(X_validation_processed)
xgboost_tuned_metrics = evaluate_predictions(y_validation, xgboost_tuned_predictions)
xgboost_tuned_train_r2 = r2_score(y_train, xgboost_tuned_train_predictions)
model_results.append({"Model": "XGBoost Tuned", **xgboost_tuned_metrics})

xgboost_tuned_summary = pd.DataFrame(
    [
        {
            **best_xgboost_params,
            "Train R2": xgboost_tuned_train_r2,
            "Validation MAE": xgboost_tuned_metrics["MAE"],
            "Validation RMSE": xgboost_tuned_metrics["RMSE"],
            "Validation R2": xgboost_tuned_metrics["R2"],
        }
    ]
)

print("Beste XGBoost-Konfiguration:")
display(xgboost_tuned_summary.round(6))
xgboost_tuning_results.round(3)


Beste XGBoost-Konfiguration:


,n_estimators,max_depth,learning_rate,Train R2,Validation MAE,Validation RMSE,Validation R2
0,200,2,0.05,0.027868,6.394587,10.205054,0.01898


,n_estimators,max_depth,learning_rate,Train R2,Validation MAE,Validation RMSE,Validation R2
0,200,2,0.05,0.028,6.395,10.205,0.019
1,150,3,0.05,0.034,6.420,10.206,0.019
2,200,3,0.05,0.037,6.423,10.211,0.018
3,200,3,0.04,0.035,6.416,10.212,0.018
4,200,3,0.03,0.032,6.413,10.213,0.017
5,250,3,0.04,0.037,6.411,10.214,0.017
6,250,3,0.05,0.040,6.411,10.216,0.017
7,300,3,0.05,0.042,6.424,10.233,0.014
8,200,4,0.03,0.041,6.419,10.236,0.013
9,300,5,0.03,0.061,6.438,10.258,0.009


## 10. CatBoost

CatBoost erhält kategoriale Merkmale nativ. Deshalb werden hier kein One-Hot-Encoding und keine sklearn-Imputation verwendet.


In [10]:
X_train_catboost = X_train[model_features].copy()
X_validation_catboost = X_validation[model_features].copy()

for column in categorical_features:
    X_train_catboost[column] = X_train_catboost[column].fillna("missing").astype(str)
    X_validation_catboost[column] = X_validation_catboost[column].fillna("missing").astype(str)

catboost_base_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
)

catboost_base_model.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_features,
)
catboost_predictions = catboost_base_model.predict(X_validation_catboost)
catboost_metrics = evaluate_predictions(y_validation, catboost_predictions)
model_results.append({"Model": "CatBoost", **catboost_metrics})

pd.DataFrame([catboost_metrics]).round(3)


,MAE,RMSE,R2
0,6.454,10.275,0.005


### CatBoost-Tuning

Die Tabelle dokumentiert eine begrenzte, vorab festgelegte Auswahl von CatBoost-Konfigurationen. `CatBoost Tuned` verwendet anschließend automatisch die Konfiguration mit dem höchsten Validation-R².


In [11]:
catboost_parameter_sets = [
    {"iterations": 200, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"iterations": 300, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"iterations": 300, "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"iterations": 300, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 5, "learning_rate": 0.02, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.025, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.02, "l2_leaf_reg": 10},
    {"iterations": 300, "depth": 6, "learning_rate": 0.015, "l2_leaf_reg": 10},
    {"iterations": 350, "depth": 6, "learning_rate": 0.015, "l2_leaf_reg": 10},
    {"iterations": 350, "depth": 6, "learning_rate": 0.0125, "l2_leaf_reg": 10},
    {"iterations": 400, "depth": 6, "learning_rate": 0.0125, "l2_leaf_reg": 10},
    {"iterations": 400, "depth": 6, "learning_rate": 0.01, "l2_leaf_reg": 10},
    {"iterations": 500, "depth": 6, "learning_rate": 0.0075, "l2_leaf_reg": 10},
    {"iterations": 600, "depth": 6, "learning_rate": 0.0075, "l2_leaf_reg": 10},
]

catboost_train_pool = Pool(
    X_train_catboost,
    y_train,
    cat_features=categorical_features,
)
catboost_validation_pool = Pool(
    X_validation_catboost,
    y_validation,
    cat_features=categorical_features,
)

catboost_tuning_rows = []

for params in catboost_parameter_sets:
    model = CatBoostRegressor(
        **params,
        loss_function="RMSE",
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
    )
    model.fit(catboost_train_pool)
    train_predictions = model.predict(catboost_train_pool)
    validation_predictions = model.predict(catboost_validation_pool)
    validation_metrics = evaluate_predictions(y_validation, validation_predictions)
    catboost_tuning_rows.append(
        {
            **params,
            "Train R2": r2_score(y_train, train_predictions),
            "Validation MAE": validation_metrics["MAE"],
            "Validation RMSE": validation_metrics["RMSE"],
            "Validation R2": validation_metrics["R2"],
        }
    )

catboost_tuning_results = (
    pd.DataFrame(catboost_tuning_rows)
    .sort_values("Validation R2", ascending=False)
    .reset_index(drop=True)
)

best_catboost_params = catboost_tuning_results.loc[
    0, ["iterations", "depth", "learning_rate", "l2_leaf_reg"]
].to_dict()
best_catboost_params["iterations"] = int(best_catboost_params["iterations"])
best_catboost_params["depth"] = int(best_catboost_params["depth"])
best_catboost_params["l2_leaf_reg"] = float(best_catboost_params["l2_leaf_reg"])

catboost_tuned_model = CatBoostRegressor(
    **best_catboost_params,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
)
catboost_tuned_model.fit(catboost_train_pool)
catboost_tuned_train_predictions = catboost_tuned_model.predict(catboost_train_pool)
catboost_tuned_predictions = catboost_tuned_model.predict(catboost_validation_pool)
catboost_tuned_metrics = evaluate_predictions(y_validation, catboost_tuned_predictions)
catboost_tuned_train_r2 = r2_score(y_train, catboost_tuned_train_predictions)
model_results.append({"Model": "CatBoost Tuned", **catboost_tuned_metrics})

catboost_tuned_summary = pd.DataFrame(
    [
        {
            **best_catboost_params,
            "Train R2": catboost_tuned_train_r2,
            "Validation MAE": catboost_tuned_metrics["MAE"],
            "Validation RMSE": catboost_tuned_metrics["RMSE"],
            "Validation R2": catboost_tuned_metrics["R2"],
        }
    ]
)

print("Beste CatBoost-Konfiguration:")
display(catboost_tuned_summary.round(6))
catboost_tuning_results.round(
    {
        "learning_rate": 4,
        "l2_leaf_reg": 3,
        "Train R2": 6,
        "Validation MAE": 3,
        "Validation RMSE": 3,
        "Validation R2": 6,
    }
)


Beste CatBoost-Konfiguration:


,iterations,depth,learning_rate,l2_leaf_reg,Train R2,Validation MAE,Validation RMSE,Validation R2
0,400,6,0.01,10.0,0.039586,6.405503,10.220418,0.016024


,iterations,depth,learning_rate,l2_leaf_reg,Train R2,Validation MAE,Validation RMSE,Validation R2
0,400,6,0.0100,10,0.039586,6.406,10.220,0.016024
1,500,6,0.0075,10,0.038838,6.411,10.223,0.015457
2,350,6,0.0125,10,0.041153,6.421,10.227,0.014788
3,300,5,0.0200,10,0.040102,6.405,10.227,0.014763
4,300,6,0.0150,10,0.041160,6.418,10.228,0.014650
5,600,6,0.0075,10,0.041353,6.421,10.229,0.014381
6,400,6,0.0125,10,0.042936,6.419,10.230,0.014110
7,350,6,0.0150,10,0.043233,6.422,10.233,0.013618
8,300,5,0.0300,10,0.045345,6.395,10.235,0.013230
9,300,6,0.0200,10,0.044965,6.416,10.237,0.012904


## 11. Modellvergleich

Die Tabelle sammelt alle Hauptmodelle und sortiert sie nach dem R² auf dem Juli-Validierungsdatensatz.


In [12]:
model_comparison = (
    pd.DataFrame(model_results)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

best_model = model_comparison.iloc[0]
second_best_model = model_comparison.iloc[1]
r2_gap_to_second = best_model["R2"] - second_best_model["R2"]

model_comparison.round(6)


,Model,MAE,RMSE,R2
0,XGBoost Tuned,6.394587,10.205054,0.018980
1,Lineare Regression,6.311167,10.207762,0.018459
2,CatBoost Tuned,6.405503,10.220418,0.016024
3,GAM Tuned,6.372980,10.245767,0.011137
4,Random Forest Tuned,6.530954,10.248715,0.010568
5,GAM,6.364469,10.249162,0.010481
6,CatBoost,6.453702,10.275033,0.005479
7,XGBoost,6.512237,10.335242,-0.006210
8,Mean Baseline,6.197273,10.339550,-0.007049
9,Random Forest,7.298178,10.636840,-0.065792


## 12. log1p-Robustheitsanalyse

Ergänzend wird geprüft, ob eine `log1p`-Transformation des Targets die Ergebnisse auf der ursprünglichen Minutenskala verbessert. Die Hauptanalyse bleibt die Modellierung von `delay_target` ohne Transformation.


In [13]:
y_train_log = np.log1p(y_train)
log1p_results = []

linear_regression_log_model = LinearRegression()
linear_regression_log_model.fit(X_train_processed, y_train_log)
linear_log_predictions = linear_regression_log_model.predict(X_validation_processed)
linear_log_predictions = np.maximum(np.expm1(linear_log_predictions), 0)
log1p_results.append(
    {"Model": "Lineare Regression log1p", **evaluate_predictions(y_validation, linear_log_predictions)}
)

gam_log_model = LinearGAM(
    f(0) + f(1) + f(2)
    + l(3)
    + s(4, n_splines=10, lam=1.0)
    + s(5, n_splines=10, lam=1.0)
    + s(6, n_splines=10, lam=1.0)
    + s(7, n_splines=10, lam=1.0)
    + s(8, n_splines=10, lam=1.0)
    + s(9, n_splines=10, lam=1.0)
    + s(10, n_splines=10, lam=1.0)
    + l(11)
)
gam_log_model.fit(X_train_gam, y_train_log)
gam_log_predictions = np.maximum(np.expm1(gam_log_model.predict(X_validation_gam)), 0)
log1p_results.append({"Model": "GAM log1p", **evaluate_predictions(y_validation, gam_log_predictions)})

random_forest_log_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
random_forest_log_model.fit(X_train_processed_dense, y_train_log)
random_forest_log_predictions = random_forest_log_model.predict(X_validation_processed_dense)
random_forest_log_predictions = np.maximum(np.expm1(random_forest_log_predictions), 0)
log1p_results.append(
    {"Model": "Random Forest log1p", **evaluate_predictions(y_validation, random_forest_log_predictions)}
)

xgboost_log_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)
xgboost_log_model.fit(X_train_processed, y_train_log)
xgboost_log_predictions = xgboost_log_model.predict(X_validation_processed)
xgboost_log_predictions = np.maximum(np.expm1(xgboost_log_predictions), 0)
log1p_results.append({"Model": "XGBoost log1p", **evaluate_predictions(y_validation, xgboost_log_predictions)})

catboost_log_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
)
catboost_log_model.fit(
    X_train_catboost,
    y_train_log,
    cat_features=categorical_features,
)
catboost_log_predictions = catboost_log_model.predict(X_validation_catboost)
catboost_log_predictions = np.maximum(np.expm1(catboost_log_predictions), 0)
log1p_results.append({"Model": "CatBoost log1p", **evaluate_predictions(y_validation, catboost_log_predictions)})

log1p_comparison = (
    pd.DataFrame(log1p_results)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

log1p_comparison.round(3)


,Model,MAE,RMSE,R2
0,Random Forest log1p,5.665,10.588,-0.056
1,XGBoost log1p,5.543,10.774,-0.093
2,CatBoost log1p,5.538,10.792,-0.097
3,GAM log1p,5.544,10.825,-0.104
4,Lineare Regression log1p,5.547,10.885,-0.116


## 13. Neutrales Fazit

Das beste Modell wird automatisch aus dem aktuellen Juli-Validation-Vergleich abgeleitet. Die log1p-Analyse bleibt davon getrennt und dient nur als Robustheitsanalyse.

Der August-Testdatensatz wurde in diesem Notebook nicht geladen oder ausgewertet.


In [14]:
print("BESTES MODELL AUF DER JULI-VALIDATION")
print(f"Model: {best_model['Model']}")
print(f"Validation MAE: {best_model['MAE']:.6f}")
print(f"Validation RMSE: {best_model['RMSE']:.6f}")
print(f"Validation R2: {best_model['R2']:.6f}")
print()
print(f"Abstand zum zweitbesten Modell ({second_best_model['Model']}): {r2_gap_to_second:.6f} R2-Punkte")

best_model_summary = pd.DataFrame(
    [
        {
            "Bestes Modell": best_model["Model"],
            "Validation MAE": best_model["MAE"],
            "Validation RMSE": best_model["RMSE"],
            "Validation R2": best_model["R2"],
            "Zweitbestes Modell": second_best_model["Model"],
            "R2-Abstand": r2_gap_to_second,
        }
    ]
)

best_model_summary.round(6)


BESTES MODELL AUF DER JULI-VALIDATION
Model: XGBoost Tuned
Validation MAE: 6.394587
Validation RMSE: 10.205054
Validation R2: 0.018980

Abstand zum zweitbesten Modell (Lineare Regression): 0.000521 R2-Punkte


,Bestes Modell,Validation MAE,Validation RMSE,Validation R2,Zweitbestes Modell,R2-Abstand
0,XGBoost Tuned,6.394587,10.205054,0.01898,Lineare Regression,0.000521
